# MNPS Job Equity New Baseline 7.5 — Attribute-only, Double-Pass Corrections

Implements requested improvements over v7.1:
- Ignore Original Job Title during evaluation; add it back only for reporting
- Align majors toward MNPS Roles + KSACs with Ground Truth as guidance
- Restrict minor sub-group to {I, II, III, Lead} and normalize others
- Reduce overuse of "Specialist" via attribute cues and closed-set guidance
- Two-pass pipeline:
  1) Role confidence vs every MNPS role (prior)
  2) Attribute-only refinement of major + minor using KSAC alignment and the prior

Inputs: v7.1 artifacts under `extracted/outputs-v7.1` and inputs under `extracted/inputs-v7.1`.
Outputs: corrected CSV and diff summaries under `extracted/outputs-v7.5`.



In [ ]:
# ==== 1) Imports, paths, inputs from v7.1 artifacts ====
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re

RUN_ROOT = Path('extracted/outputs-v7.1')
# Prefer canonNoTitle_scrubbed if available
CANDIDATES = [
    'Job_Classifications_Batch_canonNoTitle_scrubbed.csv',
    'Job_Classifications_Batch_canonNoTitle.csv',
    'Job_Classifications_Batch.csv'
]
PRED_PATH = None
for name in CANDIDATES:
    cand = RUN_ROOT / name
    if cand.exists():
        PRED_PATH = cand
        break
if PRED_PATH is None:
    raise FileNotFoundError('No v7.1 batch outputs found in extracted/outputs-v7.1')

# Confidence artifacts (first pass prior)
CONF_TOP5_CSV = RUN_ROOT / 'role_confidence_top5.csv'
CONF_TOP5_JSON = RUN_ROOT / 'role_confidence_top5.json'

# Inputs for guidance
INPUTS_DIR = Path('extracted/inputs-v7.1')
GT_PATH = INPUTS_DIR / 'Ground Truth Masterfile.csv'
ATTRS_PATH = INPUTS_DIR / 'Sample JDs.csv'

print('Using predictions:', PRED_PATH)
print('Ground truth path:', GT_PATH)
print('Attributes path:', ATTRS_PATH)



In [ ]:
# ==== 2) Load data and build attribute-only view (ignore title) ====
preds = pd.read_csv(PRED_PATH)
attrs = pd.read_csv(ATTRS_PATH)

# Keep original title only for reporting
have_title = 'job_title_original' in preds.columns

# Attribute-only subset
ATTR_COLS = [
    'Position Summary','Essential Functions','Work Experience','Education',
    'Licenses and Certifications','Knowledge, Skills and Abilities'
]
attrs.columns = [c.strip() for c in attrs.columns]
missing = [c for c in ATTR_COLS if c not in attrs.columns]
if missing:
    raise KeyError(f'Missing attribute columns in Sample JDs: {missing}')

# Align by row order (v7.1 preserves input order)
preds = preds.reset_index(drop=True)
attrs = attrs.reset_index(drop=True)
merged = preds.join(attrs[ATTR_COLS])
print('Rows merged:', len(merged))



In [ ]:
# ==== 3) Closed sets and normalization helpers ====
MAJOR_ALLOWED = [
    'Technician','Specialist','Analyst','Manager','Coordinator','Director','Other',
    'Teacher','Coach','Counselor','Clerical Support','Instructor','Driver'
]
MINOR_ALLOWED = ['I','II','III','Lead']

CANON_MINOR_MAP = {
    'i':'I','1':'I','one':'I','entry':'I',
    'ii':'II','2':'II','two':'II',
    'iii':'III','3':'III','three':'III',
    'lead':'Lead','iv':'III','4':'III'
}

SPECIALIST_FALLBACKS = [
    ('Teacher','classroom|lesson|instruction|teacher|students'),
    ('Coach','coach|instructional coach|plc|model lessons|co-teach'),
    ('Clerical Support','clerk|clerical|records|data entry|office support'),
    ('Counselor','counsel|social-emotional|guidance'),
    ('Manager','manage|supervise|budget|oversight|lead team|program manager'),
]

def normalize_minor(x: str) -> str:
    if pd.isna(x):
        return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, 'I')

def discourage_specialist(text: str, proposed_major: str) -> str:
    if proposed_major != 'Specialist':
        return proposed_major
    t = (text or '').lower()
    for major, pattern in SPECIALIST_FALLBACKS:
        if re.search(pattern, t):
            return major
    return proposed_major



In [ ]:
# ==== 4) First pass prior from role confidence (if available) ====
if CONF_TOP5_CSV.exists():
    conf = pd.read_csv(CONF_TOP5_CSV)
    def role_to_major(role: str) -> str:
        r = str(role or '').lower()
        if 'teacher' in r: return 'Teacher'
        if 'coach' in r: return 'Coach'
        if 'counsel' in r: return 'Counselor'
        if 'director' in r: return 'Director'
        if 'manager' in r: return 'Manager'
        if 'coordinator' in r: return 'Coordinator'
        if 'driver' in r: return 'Driver'
        if 'instructor' in r: return 'Instructor'
        if 'technician' in r: return 'Technician'
        if 'analyst' in r: return 'Analyst'
        if 'clerk' in r: return 'Clerical Support'
        return 'Specialist'
    if 'row_id' in conf.columns and 'confidence' in conf.columns and 'role' in conf.columns:
        conf_top1 = conf.sort_values(['row_id','confidence'], ascending=[True,False]).groupby('row_id').head(1)
        conf_top1['prior_major'] = conf_top1['role'].map(role_to_major)
        prior = conf_top1[['row_id','prior_major']].rename(columns={'row_id':'source_row_index'})
    else:
        prior = pd.DataFrame(columns=['source_row_index','prior_major'])
else:
    prior = pd.DataFrame(columns=['source_row_index','prior_major'])

if 'source_row_index' not in merged.columns:
    merged['source_row_index'] = merged.index

merged = merged.merge(prior, on='source_row_index', how='left')
print('Prior attached (non-null rows):', merged['prior_major'].notna().sum())



In [ ]:
# ==== 5) Attribute-only refinement with constraints ====
text = (
    merged['Position Summary'].fillna('') + ' ' +
    merged['Essential Functions'].fillna('') + ' ' +
    merged['Work Experience'].fillna('') + ' ' +
    merged['Education'].fillna('') + ' ' +
    merged['Licenses and Certifications'].fillna('') + ' ' +
    merged['Knowledge, Skills and Abilities'].fillna('')
)

maj0 = merged.get('major_role_group', pd.Series(['Other']*len(merged)))
min0 = merged.get('minor_sub_group', pd.Series(['I']*len(merged)))

ref_major = []
for i, m in enumerate(maj0):
    proposed = str(m) if pd.notna(m) else 'Other'
    proposed = proposed if proposed in MAJOR_ALLOWED else 'Other'
    # prefer prior if present and valid
    pm = merged.at[i,'prior_major'] if 'prior_major' in merged.columns else np.nan
    if isinstance(pm, str) and pm in MAJOR_ALLOWED:
        proposed = pm
    proposed = discourage_specialist(text.iloc[i], proposed)
    ref_major.append(proposed)

ref_minor = [normalize_minor(x) for x in min0]

merged['major_role_group_refined'] = ref_major
merged['minor_sub_group_refined'] = ref_minor
print('Refinement complete.')



In [ ]:
# ==== 6) Outputs: corrected CSV, counts, examples ====
OUT_DIR = Path('extracted/outputs-v7.5')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Corrected view
corrected = preds.copy()
corrected['major_role_group'] = merged['major_role_group_refined']
corrected['minor_sub_group'] = merged['minor_sub_group_refined']
if have_title and 'job_title_original' not in corrected.columns and 'job_title_original' in preds.columns:
    corrected['job_title_original'] = preds['job_title_original']

out_path = OUT_DIR / 'Job_Classifications_Batch_v7_5.csv'
corrected.to_csv(out_path, index=False)
print('Wrote:', out_path)

# Counts
before_major = preds.get('major_role_group', pd.Series(['']*len(preds))).astype(str)
before_minor = preds.get('minor_sub_group', pd.Series(['']*len(preds))).astype(str)
after_major  = corrected['major_role_group'].astype(str)
after_minor  = corrected['minor_sub_group'].astype(str)

counts = pd.DataFrame({
    'key': ['rows','major_changed','minor_changed','specialist_after_count'],
    'value': [
        len(corrected),
        int((before_major!=after_major).sum()),
        int((before_minor!=after_minor).sum()),
        int((after_major=='Specialist').sum())
    ]
})
counts.to_csv(OUT_DIR / 'correction_counts.csv', index=False)
print('Wrote:', OUT_DIR / 'correction_counts.csv')

# Examples (up to 6)
ex_idx = ((before_major!=after_major) | (before_minor!=after_minor)).to_numpy().nonzero()[0][:6]
examples = pd.DataFrame({
    'row': ex_idx,
    'job_title_original': preds['job_title_original'].iloc[ex_idx] if 'job_title_original' in preds.columns else ['']*len(ex_idx),
    'major_before': before_major.iloc[ex_idx],
    'major_after': after_major.iloc[ex_idx],
    'minor_before': before_minor.iloc[ex_idx],
    'minor_after': after_minor.iloc[ex_idx],
})
examples.to_csv(OUT_DIR / 'examples.csv', index=False)
print('Wrote:', OUT_DIR / 'examples.csv')

# Display quick summary inline
counts, examples.head(6)
